<div dir="rtl">

<h1>«معاكِ»: نماذج التحاليل والأشعة (النسخة المراجَعة)</h1>

<p>النوت بوك ده فيه نموذجين منفصلين، وكل نموذج متقيّم بطريقة سليمة:</p>
<ul>
<li><b>الجزء أ، تحاليل الدم (Breast Cancer Coimbra):</b> هل القيم دي أقرب لقيم مريضات سرطان الثدي ولا لقيم الستات السليمات؟</li>
<li><b>الجزء ب، صور الأشعة (BCMID من الإسكندرية):</b> هل تقييم الحالة في الأشعة BI-RADS 4 أو أعلى، يعني محتاجة عيّنة؟ بالماموجرام لوحده، وبالسونار لوحده، وبالاتنين مع بعض.</li>
<li><b>الجزء ج:</b> ليه شلنا الدمج بين التحاليل والصور.</li>
<li><b>الجزء د:</b> أسئلة متوقعة من الدكتور وإجاباتها.</li>
</ul>

<p><b>قاعدة النوت بوك:</b> كل رقم بيظهر هنا محسوب بالكود، ومفيش ولا رقم مكتوب باليد.</p>
<p><b>التشغيل:</b> Runtime ← Run all. لو فيه صور ميزاتها لسه ما اتحسبتش، اختاري GPU من Runtime ← Change runtime type علشان الاستخراج يبقى أسرع. ولو كل الميزات محفوظة، الـ CPU العادي كفاية.</p>

</div>

<div dir="rtl">

<h2>إيه اللي اتغير عن النسخة القديمة؟</h2>

<table>
<tr><th>#</th><th>المشكلة في النسخة القديمة</th><th>ليه دي مشكلة</th><th>الحل هنا</th></tr>
<tr><td>1</td><td>أرقام مكتوبة باليد: <code>81.20%</code> و<code>88.50%</code>، و<code>max(..., 91.80)</code>، و<code>+ 4.1</code></td><td>دي مش نتايج النموذج، وعرضها كنتيجة غلط علمي كبير</td><td>اتشالت، وكل رقم دلوقتي محسوب من التقييم</td></tr>
<tr><td>2</td><td>تصنيفات عشوائية لصور BCMID (<code>np.random.choice</code>)</td><td>النموذج اتعلم ضوضاء، وعلشان كده دقته كانت 45.83%، أقل من رمي عملة</td><td>التصنيفات الحقيقية من <code>BCMID_annotations.tsv</code></td></tr>
<tr><td>3</td><td>دمج تحاليل Coimbra (البرتغال) مع صور BCMID (مصر)</td><td>الدمج محتاج تحاليل وصور لنفس المريضة، وهنا دول ستات مختلفين</td><td>الدمج دلوقتي بين الماموجرام والسونار لنفس الحالة جوه BCMID</td></tr>
<tr><td>4</td><td>نموذج الدمج اتدرّب واتقيّم على نفس الـ 24 عيّنة</td><td>التقييم على بيانات النموذج شافها بيدي نتيجة أعلى من الحقيقة</td><td>كل تقييم على بيانات النموذج ما شافهاش في التدريب</td></tr>
<tr><td>5</td><td>عرض أعلى fold بس (78.26%)</td><td>أعلى رقم من 5 محاولات ده حظ، والمتوسط أقل</td><td>المتوسط ± الانحراف المعياري على 50 تقسيمة</td></tr>
<tr><td>6</td><td><code>PolynomialFeatures</code> و<code>SelectKBest</code> و<code>StandardScaler</code> اتعملوا على كل البيانات قبل التقسيم</td><td>تسريب بيانات (data leakage): اختيار الميزات شاف تصنيفات بيانات الاختبار</td><td>كل الخطوات جوه <code>Pipeline</code> بتتعلم من بيانات التدريب بس</td></tr>
<tr><td>7</td><td>كل صور السونار اتجمعت تحت اسم واحد هو <code>Ultrasound</code></td><td>غلطة في الكود: واحدة من الـ 208 «مريضة» وهمية، وفيها سونار كل الحالات</td><td>رقم الحالة بيتقري من اسم الفولدر</td></tr>
<tr><td>8</td><td>متوسط واحد بيخلط الماموجرام بالسونار</td><td>نوعين مختلفين تمامًا من الصور</td><td>نموذج لكل نوع، وبعدين دمج الاحتمالات</td></tr>
<tr><td>9</td><td>نفس المريضة ممكن تظهر في التدريب وفي الاختبار</td><td>النموذج بيتعرف على المريضة بدل المرض</td><td>التقسيم حسب المريضة (<code>StratifiedGroupKFold</code>)</td></tr>
<tr><td>10</td><td>صورة فاضية (0 بايت): <code>0165y09_US_1.png</code></td><td>الملف بايظ في الداتا نفسها</td><td>بنتخطاها ونسجلها</td></tr>
<tr><td>11</td><td>صور الماموجرام في BCMID جاية من جهازين: جهاز بيحفظها معكوسة (الخلفية بيضا) وجهاز عادي (الخلفية سودا)، والثدي اليمين في عكس اتجاه الشمال</td><td>لو الصور مش بشكل واحد، النموذج بيتعلم شكل الجهاز بدل شكل النسيج</td><td>بنعرف الخلفية من الصورة نفسها، ونعكس المعكوس، ونخلي جدار الصدر على الشمال دايمًا، ونقص منطقة الثدي</td></tr>
<tr><td>12</td><td>الدرايف كان فيه 1643 صورة بس، والداتا كاملة 2001 صورة</td><td>مساحة جوجل المجانية (15 GB) مكفتش لباقي الصور</td><td>قايمة الصور الرسمية من ملف Zenodo نفسه، وميزات الصور الناقصة اتحسبت مرة واحدة من الملف ده بنفس الكود، فكل الصور داخلة دلوقتي</td></tr>
<tr><td>13</td><td>مكانش فيه أي اختبار إن النموذج مش بيعتمد على اختصار</td><td>في BCMID الحالات من 2020 وبعدها نسبة الـ Positive فيها أعلى بكتير، فالسنة لوحدها بتتوقع التصنيف، والنموذج ممكن يتعلم السنة (من شكل الجهاز مثلًا) بدل المرض</td><td>ب-7-ب: الـ AUC بين حالات من نفس السنة بس، ونموذج مابيشوفش الصور خالص، والماموجرام من غير الصور الإضافية</td></tr>
</table>

</div>

<div dir="rtl">
<h2>0) التجهيز</h2>
<p>نربط Google Drive، ونحدد أماكن الملفات. الدالة <code>find_file</code> تبحث عن الملف بالاسم في Colab وفي Drive، علشان مايهمش رفعتيه فين بالظبط.</p>
<p>الملفات المطلوبة على Drive: <code>dataR2.csv</code> و<code>BCMID_annotations.tsv</code> وفولدر <code>BCMID</code> داخل MyDrive. ولو الدرايف مكفاش كل الصور، دفتر <code>maaki_bcmid_zenodo_features.ipynb</code> بيحسب ميزات الصور الناقصة من ملف Zenodo، ويحفظها مع قايمة الصور الرسمية في <code>maaki_outputs</code> (التفاصيل في ب-1).</p>
</div>

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ربط Google Drive (لو الكود شغال على Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE = os.environ.get('MAAKI_DRIVE', '/content/drive/MyDrive')
OUT_DIR = os.path.join(DRIVE, 'maaki_outputs')      # هنا بنحفظ الميزات والنماذج
os.makedirs(OUT_DIR, exist_ok=True)


def find_file(name, roots=('/content', DRIVE)):
    # يبحث عن ملف بالاسم، ويتخطى فولدر BCMID لأنه كبير
    for root in roots:
        if os.path.isfile(os.path.join(root, name)):
            return os.path.join(root, name)
    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in ('BCMID', 'drive', 'maaki_outputs', 'sample_data', '.config')]
            if name in filenames:
                return os.path.join(dirpath, name)
    return None


COIMBRA_CSV = find_file('dataR2.csv')
BCMID_TSV = find_file('BCMID_annotations.tsv')
BCMID_DIR = os.path.join(DRIVE, 'BCMID')

print('dataR2.csv           ->', COIMBRA_CSV)
print('BCMID_annotations.tsv ->', BCMID_TSV)
print('BCMID folder          ->', BCMID_DIR, '(موجود)' if os.path.isdir(BCMID_DIR) else '(مش موجود)')
if COIMBRA_CSV is None or BCMID_TSV is None:
    print('\n⚠️ ارفعي الملف الناقص على Drive داخل MyDrive، وبعدين شغّلي الخلية دي تاني.')

<div dir="rtl">

<h1>الجزء أ: تحاليل الدم (Breast Cancer Coimbra)</h1>

<p><b>عن البيانات:</b> 116 ست من كويمبرا في البرتغال: 64 مريضة سرطان ثدي و52 ست سليمة (مجموعة ضابطة). المصدر UCI Machine Learning Repository، والرخصة CC BY 4.0. والورقة الأصلية: Patrício وزملاؤه، «Using Resistin, glucose, age and BMI to predict the presence of breast cancer»، مجلة BMC Cancer سنة 2018.</p>

<table>
<tr><th>العمود</th><th>المعنى</th><th>الوحدة</th></tr>
<tr><td>Age</td><td>السن</td><td>سنة</td></tr>
<tr><td>BMI</td><td>مؤشر كتلة الجسم</td><td>kg/m²</td></tr>
<tr><td>Glucose</td><td>سكر الدم</td><td>mg/dL</td></tr>
<tr><td>Insulin</td><td>الإنسولين</td><td>µU/mL</td></tr>
<tr><td>HOMA</td><td>مؤشر مقاومة الإنسولين، بيتحسب من السكر والإنسولين</td><td>من غير وحدة</td></tr>
<tr><td>Leptin</td><td>هرمون بتفرزه الدهون</td><td>ng/mL</td></tr>
<tr><td>Adiponectin</td><td>هرمون بتفرزه الدهون</td><td>µg/mL</td></tr>
<tr><td>Resistin</td><td>هرمون من الدهون مرتبط بالالتهاب ومقاومة الإنسولين</td><td>ng/mL</td></tr>
<tr><td>MCP.1</td><td>بروتين بيجذب خلايا المناعة (MCP-1)</td><td>pg/dL</td></tr>
<tr><td>Classification</td><td>1 = سليمة، 2 = مريضة</td><td></td></tr>
</table>

<p><b>تنبيه:</b> دي مش تحاليل CBC (صورة دم كاملة). دي تحاليل سكر وهرمونات بتفرزها الدهون، وبعضها (Leptin وAdiponectin وResistin وMCP-1) مش بيتطلب بشكل روتيني في مصر.</p>

</div>

In [ ]:
df = pd.read_csv(COIMBRA_CSV)
FEATURES = ['Age', 'BMI', 'Glucose', 'Insulin', 'HOMA', 'Leptin', 'Adiponectin', 'Resistin', 'MCP.1']
df['target'] = (df['Classification'] == 2).astype(int)      # 1 = مريضة، 0 = سليمة

print('عدد الصفوف والأعمدة:', df.shape)
print('عدد القيم الناقصة:', int(df[FEATURES].isnull().sum().sum()))
print(df['target'].value_counts().rename({1: 'مريضة (1)', 0: 'سليمة (0)'}))
display(df[FEATURES].describe().T.round(2))

<div dir="rtl">
<h3>أ-1) نظرة على البيانات</h3>
<p>الرسومات بتقارن كل تحليل بين المجموعتين. والجدول بيقيس كل تحليل لوحده بالـ AUC: 0.5 يعني التحليل ده لوحده مش بيفرّق، و1 يعني بيفرّق تمامًا. الجدول ده للوصف بس، ومش بنستخدمه في اختيار النموذج.</p>
<p>(عناوين الرسومات بالإنجليزي لأن مكتبة matplotlib لا تكتب العربي بشكل صحيح.)</p>
</div>

In [ ]:
from sklearn.metrics import roc_auc_score

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, col in zip(axes.ravel(), FEATURES):
    ax.boxplot([df.loc[df['target'] == 0, col], df.loc[df['target'] == 1, col]])
    ax.set_xticks([1, 2], ['healthy', 'cancer'])
    ax.set_title(col)
plt.tight_layout()
plt.show()

single = []
for col in FEATURES:
    auc = roc_auc_score(df['target'], df[col])
    single.append({'التحليل': col, 'AUC لوحده': round(max(auc, 1 - auc), 3),
                   'الاتجاه': 'أعلى في المريضات' if auc >= 0.5 else 'أقل في المريضات'})
display(pd.DataFrame(single).sort_values('AUC لوحده', ascending=False).reset_index(drop=True))

<div dir="rtl">

<h3>أ-2) طريقة التقييم</h3>
<ul>
<li><b>ليه مش تقسيمة واحدة 80/20؟</b> عندنا 116 حالة بس، يعني الاختبار هيبقى 24 حالة، والنتيجة هتتغير كتير حسب الحظ. علشان كده بنستخدم Repeated Stratified K-Fold: 5 أجزاء × 10 مرات = 50 تقييم، وكل حالة بتدخل الاختبار 10 مرات.</li>
<li><b>Stratified:</b> كل جزء فيه تقريبًا نفس نسبة المريضات.</li>
<li><b>Pipeline:</b> التطبيع (StandardScaler) بيتعلم من بيانات التدريب بس في كل مرة، فمفيش تسريب.</li>
<li><b>المقاييس:</b>
<ul>
<li>AUC: قدرة النموذج يدي المريضة احتمال أعلى من السليمة. 0.5 = عشوائي، و1 = مثالي.</li>
<li>Sensitivity: نسبة المريضات اللي النموذج اكتشفهم.</li>
<li>Specificity: نسبة السليمات اللي النموذج قال عليهم سليمات صح.</li>
<li>Balanced accuracy: متوسط الاتنين اللي فوق.</li>
</ul></li>
<li><b>مجموعتين من الميزات:</b> كل التحاليل (9)، و4 ميزات اختارتها الورقة الأصلية قبلنا (Glucose وResistin وAge وBMI).</li>
</ul>
<p>بنعرض المتوسط ± الانحراف المعياري. الخلية دي بتاخد حوالي دقيقة.</p>

</div>

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import make_scorer, recall_score

X_all, y = df[FEATURES], df['target']
FOUR = ['Glucose', 'Resistin', 'Age', 'BMI']


def make_svm():
    return Pipeline([('scale', StandardScaler()), ('clf', SVC(kernel='rbf', probability=True, random_state=SEED))])


MODELS = {
    'Logistic Regression': lambda: Pipeline([('scale', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'SVM (RBF)': make_svm,
    'Random Forest': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED),
    'Gradient Boosting': lambda: GradientBoostingClassifier(random_state=SEED),
}
SCORING = {'AUC': 'roc_auc',
           'Sensitivity': make_scorer(recall_score, pos_label=1),
           'Specificity': make_scorer(recall_score, pos_label=0),
           'Balanced acc.': 'balanced_accuracy',
           'Accuracy': 'accuracy'}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

rows = []
for name, make in MODELS.items():
    for set_name, cols in [('كل التحاليل (9)', FEATURES), ('4 (من الورقة)', FOUR)]:
        r = cross_validate(make(), X_all[cols], y, cv=cv, scoring=SCORING, n_jobs=-1)
        row = {'model': name, 'features': set_name}
        for k in SCORING:
            row[k] = r[f'test_{k}'].mean()
            row[k + ' sd'] = r[f'test_{k}'].std()
        rows.append(row)

tab_results = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
show = tab_results[['model', 'features']].copy()
for k in SCORING:
    show[k] = tab_results[k].map('{:.2f}'.format) + ' ± ' + tab_results[k + ' sd'].map('{:.2f}'.format)
display(show)

<div dir="rtl">

<h3>أ-3) اختيار النموذج</h3>
<p>اخترنا <b>SVM بالـ 4 ميزات</b> (Glucose وResistin وAge وBMI)، والأسباب:</p>
<ol>
<li>الميزات دي اختارتها دراسة منشورة قبلنا، مش احنا اللي اخترناها بعد ما شفنا النتايج. ده بيقلل التفاؤل الزايد اللي بيحصل لما نجرب نماذج كتير ونختار أحسنها.</li>
<li>نموذج بسيط، وتحاليله أقل، فأسهل يتطبق.</li>
<li>غالبًا هيطلع في أول الجدول. لو جه نموذج تاني فوقه بفرق صغير جوه الانحراف المعياري، فده مش فرق حقيقي.</li>
</ol>
<p><b>مقارنة بالورقة الأصلية</b> (Table 4، نموذج SVM بنفس الأربع ميزات): AUC بين 0.87 و0.91، وSensitivity بين 0.82 و0.88، وSpecificity بين 0.84 و0.90 (فترات ثقة 95%). قارني الأرقام دي بصف «SVM (RBF)، 4 (من الورقة)» في الجدول اللي فوق.</p>
<p><b>ملاحظة صريحة:</b> احنا بنقارن 8 تجارب على نفس التقسيمات، فالرقم الأعلى ممكن يكون متفائل شوية. الحل الأدق هو nested cross-validation، وده خطوة لاحقة.</p>

</div>

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, roc_curve, ConfusionMatrixDisplay

# كل حالة بتاخد احتمال من نموذج ما شافهاش في التدريب (out-of-fold)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_tab = cross_val_predict(make_svm(), df[FOUR], y, cv=cv5, method='predict_proba')[:, 1]
pred_tab = (oof_tab >= 0.5).astype(int)
cm = confusion_matrix(y, pred_tab)
tn, fp, fn, tp = cm.ravel()
print(f'Sensitivity = {tp / (tp + fn):.2f}   ({tp} من {tp + fn} مريضة اتكشفت)')
print(f'Specificity = {tn / (tn + fp):.2f}   ({tn} من {tn + fp} سليمة اتعرفت صح)')
print(f'AUC         = {roc_auc_score(y, oof_tab):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(cm, display_labels=['healthy', 'cancer']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('SVM, 4 features (out-of-fold)')
fpr, tpr, _ = roc_curve(y, oof_tab)
axes[1].plot(fpr, tpr, label=f'SVM, 4 features (AUC = {roc_auc_score(y, oof_tab):.2f})')
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='random')
axes[1].set_xlabel('1 - specificity')
axes[1].set_ylabel('sensitivity')
axes[1].legend()
plt.tight_layout()
plt.show()

<div dir="rtl">
<h3>أ-4) أنهي تحليل مؤثر أكتر؟ (Permutation importance)</h3>
<p>الفكرة: بنخلط قيم عمود واحد عشوائيًا في بيانات الاختبار، ونشوف الـ AUC نزل قد إيه. لو نزل كتير يبقى النموذج معتمد على العمود ده. بنعمل ده جوه كل fold، على نموذج SVM بكل التحاليل التسعة.</p>
</div>

In [ ]:
from sklearn.inspection import permutation_importance

drops = []
for tr, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_all, y):
    m = make_svm().fit(X_all.iloc[tr], y.iloc[tr])
    r = permutation_importance(m, X_all.iloc[te], y.iloc[te], scoring='roc_auc', n_repeats=20, random_state=SEED)
    drops.append(r.importances_mean)
importance = pd.Series(np.mean(drops, axis=0), index=FEATURES).sort_values()
importance.plot.barh(figsize=(7, 4), title='Drop in AUC when the feature is shuffled')
plt.tight_layout()
plt.show()

<div dir="rtl">
<h3>أ-5) ليه رقم 78.26% القديم كان متفائل؟</h3>
<p>الخلية دي بتشغّل الطريقة القديمة بالظبط (تحويل البيانات كلها قبل التقسيم، و5 أجزاء، وعرض أعلى جزء). هتلاحظي إن أعلى fold أعلى من المتوسط بفرق واضح، وإن الطريقة المعقدة دي مش أحسن من SVM البسيط.</p>
</div>

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import StackingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

old_X = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False).fit_transform(X_all)
old_X = SelectKBest(f_classif, k=12).fit_transform(old_X, y)       # هنا التسريب: شاف كل التصنيفات
old_X = StandardScaler().fit_transform(old_X)
old_model = StackingClassifier(
    estimators=[('rf', RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)),
                ('et', ExtraTreesClassifier(n_estimators=100, max_depth=4, random_state=42)),
                ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)),
                ('knn', KNeighborsClassifier(n_neighbors=5))],
    final_estimator=LogisticRegression(C=0.5, random_state=42), cv=5)
old_scores = cross_val_score(old_model, old_X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring='accuracy')
print('دقة كل fold:', np.round(old_scores, 3))
print(f'أعلى fold = {old_scores.max():.3f}   ← ده الرقم اللي كان بيتعرض')
print(f'المتوسط   = {old_scores.mean():.3f} ± {old_scores.std():.3f}   ← ده الرقم الصح للطريقة دي')

<div dir="rtl">
<h3>أ-6) النموذج النهائي وحفظه</h3>
<p>بعد ما خلصنا التقييم، بندرّب النموذج المختار على الـ 116 حالة كلهم، ونحفظه علشان السيرفر يستخدمه. الأرقام اللي بنقولها عن أداءه هي أرقام الـ cross-validation اللي فوق، مش أداءه على بيانات اتدرب عليها.</p>
<p>الناتج احتمال: «القيم دي شبه قيم مجموعة المريضات قد إيه». ده مؤشر للطبيب، مش تشخيص.</p>
</div>

In [ ]:
import joblib
from datetime import date

final_tab = make_svm().fit(df[FOUR], y)
joblib.dump(final_tab, os.path.join(OUT_DIR, 'coimbra_svm_4features.joblib'))

chosen = tab_results[(tab_results['model'] == 'SVM (RBF)') & (tab_results['features'] == '4 (من الورقة)')].iloc[0]
tab_meta = {
    'model': 'SVM (RBF) inside Pipeline(StandardScaler)',
    'features_in_order': FOUR,
    'units': {'Glucose': 'mg/dL', 'Resistin': 'ng/mL', 'Age': 'years', 'BMI': 'kg/m2'},
    'output': 'probability of belonging to the breast-cancer group (not a diagnosis)',
    'evaluation': 'RepeatedStratifiedKFold 5x10',
    'cv_metrics': {k: [round(float(chosen[k]), 3), round(float(chosen[k + ' sd']), 3)] for k in SCORING},
    'data': 'Breast Cancer Coimbra (UCI, CC BY 4.0), 116 women, Coimbra, Portugal',
    'trained_on': str(date.today()),
}
with open(os.path.join(OUT_DIR, 'coimbra_model.json'), 'w', encoding='utf-8') as f:
    json.dump(tab_meta, f, ensure_ascii=False, indent=2)


def predict_tab(age, bmi, glucose, resistin):
    x = pd.DataFrame([{'Glucose': glucose, 'Resistin': resistin, 'Age': age, 'BMI': bmi}])[FOUR]
    return float(final_tab.predict_proba(x)[0, 1])


print('اتحفظ في:', OUT_DIR)
# مثال: الوسيط (median) لكل مجموعة، والاحتمال اللي النموذج بيديه له
for label, name in [(0, 'وسيط السليمات'), (1, 'وسيط المريضات')]:
    med = df.loc[df['target'] == label, FOUR].median()
    p = predict_tab(med['Age'], med['BMI'], med['Glucose'], med['Resistin'])
    values = '، '.join(f'{k} {v:.1f}' for k, v in med.items())
    print(f'{name} ({values}) ← الاحتمال = {p:.2f}')

<div dir="rtl">
<h3>أ-7) حدود النموذج ده (لازم تقوليها)</h3>
<ul>
<li>116 حالة بس، من مركز واحد في البرتغال. محتاج اختبار على بيانات تانية قبل أي استخدام.</li>
<li>المقارنة بين مريضات متشخصات وسيدات سليمات، مش فحص مبكر لسيدات لسه مش عارفين حالتهم.</li>
<li>Resistin مش تحليل روتيني، فأغلب المريضات مش هيبقى معاهم القيمة دي.</li>
<li>الناتج مؤشر للطبيب، مش تشخيص.</li>
</ul>
</div>

<div dir="rtl">

<h1>الجزء ب: صور الأشعة (BCMID)</h1>

<p><b>عن البيانات:</b> BCMID (Breast Cancer Multimodal Imaging Dataset) من مستشفى أيادي المستقبل في الإسكندرية، وبتتوصف بإنها أول داتا مصرية متعددة الوسائط لسرطان الثدي. الرخصة CC BY 4.0، والـ DOI هو 10.5281/zenodo.19402690.</p>
<ul>
<li>332 حالة من 323 مريضة: 9 مريضات ليهم حالتين (مثلًا أشعة في سنتين مختلفتين).</li>
<li>كل حالة فيها فولدر Mammogram وفولدر Ultrasound وتقرير PDF. حسب ورقة منشورة على نفس الداتا: 951 صورة ماموجرام و1050 صورة سونار.</li>
<li>رقم المريضة هو الرقم اللي قبل حرف y في اسم الحالة، و y + رقمين = السنة. مثال: <code>1000y16</code> و<code>1000y17</code> نفس المريضة.</li>
</ul>

<p><b>التصنيف:</b> كل حالة ليها BI-RADS من طبيب الأشعة، مش نتيجة عيّنة (باثولوجي). عمود Pathology Label في الملف مأخوذ من BI-RADS مباشرة، والجدول اللي تحت بيثبت ده. يعني النموذج بيتعلم تقييم طبيب الأشعة.</p>
<ul>
<li><b>Positive</b> = BI-RADS 4 و5 و6 (اشتباه يحتاج عيّنة، أو سرطان متأكد قبل كده).</li>
<li><b>Negative</b> = BI-RADS 1 و2 و3.</li>
<li>BI-RADS 0 (5 حالات) = التقييم مش مكتمل، فبنشيله.</li>
</ul>

</div>

In [ ]:
ann = pd.read_csv(BCMID_TSV, sep='\t').rename(columns={
    'Patient ID': 'case', 'BI-RADS Label': 'birads', 'Binary Label': 'binary', 'Pathology Label': 'pathology'})
ann['case'] = ann['case'].astype(str)
ann['patient'] = ann['case'].str.extract(r'^(\d+)y')[0].astype(int)     # رقم المريضة

print('عدد الحالات:', len(ann), '| عدد المريضات:', ann['patient'].nunique())
print('\nBI-RADS مقابل Pathology Label (كل BI-RADS ليه تصنيف واحد بس):')
display(pd.crosstab(ann['birads'], ann['pathology'].fillna('—')))
print('مريضات ليهم أكتر من حالة:')
display(ann[ann.duplicated('patient', keep=False)].sort_values('patient')[['case', 'patient', 'birads']])

ann = ann[ann['birads'] != 0].copy()
ann['y'] = (ann['birads'] >= 4).astype(int)
print(ann['y'].value_counts().rename({0: 'Negative (BI-RADS 1-3)', 1: 'Positive (BI-RADS 4-6)'}))
print(f"\nلو قلنا «Negative» على كل الحالات: Accuracy = {(ann['y'] == 0).mean():.2f} ومن غير ما نكتشف ولا حالة.")
print('علشان كده الـ Accuracy لوحدها مقياس مضلل هنا.')

<div dir="rtl">
<h3>ب-1) فهرس الصور</h3>
<p>نوع الصورة بيتقري من اسم الفولدر الفرعي (Mammogram أو Ultrasound)، ورقم الحالة من اسم الفولدر الرئيسي. ده بيصلح الخطأ القديم اللي كان بيجمّع كل صور السونار تحت اسم واحد.</p>
<p><b>قايمة الصور:</b> مساحة جوجل المجانية (15 GB) مكفتش لكل صور الماموجرام، فجزء منها مش على الدرايف. علشان كده القايمة بتيجي من <code>bcmid_manifest.csv</code>، وده الفهرس الرسمي لكل صور الداتا، معمول من ملف Zenodo نفسه. الصور اللي مش على الدرايف اتحسبت ميزاتها مرة واحدة في دفتر <code>maaki_bcmid_zenodo_features.ipynb</code>، بنفس دوال التجهيز ونفس ResNet50 بالحرف، واتحفظت مع باقي الميزات. ولو ملف القايمة مش موجود، الفهرس بيتعمل من فولدرات الدرايف زي الأول.</p>
</div>

In [ ]:
IMG_EXT = ('.png', '.jpg', '.jpeg')
rows = []
for case in sorted(os.listdir(BCMID_DIR)) if os.path.isdir(BCMID_DIR) else []:
    for modality in ('Mammogram', 'Ultrasound'):
        mod_dir = os.path.join(BCMID_DIR, case, modality)
        if not os.path.isdir(mod_dir):
            continue
        for name in sorted(os.listdir(mod_dir)):
            if name.lower().endswith(IMG_EXT):
                path = os.path.join(mod_dir, name)
                rows.append({'case': case, 'modality': modality, 'file': name, 'path': path,
                             'bytes': os.path.getsize(path)})
drive_files = pd.DataFrame(rows, columns=['case', 'modality', 'file', 'path', 'bytes'])   # الصور اللي على الدرايف

MANIFEST = os.path.join(OUT_DIR, 'bcmid_manifest.csv')
if os.path.exists(MANIFEST):
    # الفهرس الرسمي لكل صور الداتا (معمول من ملف Zenodo نفسه)
    images = pd.read_csv(MANIFEST, dtype={'case': str, 'modality': str, 'file': str})
    images['path'] = [os.path.join(BCMID_DIR, c, m, f)
                      for c, m, f in zip(images['case'], images['modality'], images['file'])]
    images['on_drive'] = images['path'].isin(drive_files['path'])
    print('قايمة الصور من الفهرس الرسمي (bcmid_manifest.csv)')
else:
    images = drive_files.assign(on_drive=True)
    print('قايمة الصور من فولدرات الدرايف')
images['side'] = images['file'].str.extract(r'_(L|R)_')[0]
images['view'] = images['file'].str.extract(r'_(CC|MLO)_')[0]

EXPECTED = {'Mammogram': 951, 'Ultrasound': 1050}
for modality, grp in images.groupby('modality'):
    n, d = len(grp), int(grp['on_drive'].sum())
    note = '' if n >= EXPECTED[modality] else f'   ⚠️ ناقص حوالي {EXPECTED[modality] - n} صورة: اتأكدي إن فولدر BCMID اترفع كامل'
    where = '' if d == n else f' (منهم {d} على الدرايف، و{n - d} مش على الدرايف وبنستخدم ميزاتهم المحفوظة)'
    print(f'{modality}: {n} صورة{where}{note}')

empty = images[images['bytes'] == 0]
print('\nصور فاضية (0 بايت) هنتخطاها:', empty['file'].tolist())
images = images[images['bytes'] > 0].reset_index(drop=True)

no_images = sorted(set(ann['case']) - set(images['case']))
print(f'حالات ليها تصنيف ومفيش ليها صور هنا: {len(no_images)}')
print(f"حالات فيها ماموجرام: {images.loc[images['modality'] == 'Mammogram', 'case'].nunique()}، "
      f"وفيها سونار: {images.loc[images['modality'] == 'Ultrasound', 'case'].nunique()}")

<div dir="rtl">
<h3>ب-2) تجهيز الصور</h3>
<ul>
<li><b>الماموجرام:</b> صور BCMID جاية من جهازين. جهاز بيحفظ الصورة معكوسة (خلفية بيضا ونسيج غامق)، وجهاز تاني بالشكل المعتاد (خلفية سودا ونسيج فاتح). وكمان الثدي اليمين بيبقى في عكس اتجاه الشمال. علشان كل الصور تبقى بشكل واحد:
<ol>
<li>بنعرف الخلفية: هي الجنب (يمين أو شمال الصورة) اللي لونه ثابت ومفيهوش نسيج.</li>
<li>لو الخلفية بيضا، بنعكس الألوان.</li>
<li>بنقلب الصورة لو محتاجة، علشان جدار الصدر يبقى على الشمال في كل الصور.</li>
<li>بنقص منطقة الثدي (أكبر جزء فاتح في الصورة)، فبنشيل الخلفية الفاضية وحرف L أو R.</li>
<li>بنكمّل الصورة لمربع من غير ما نمطّها، ونصغّرها لـ 224×224.</li>
</ol></li>
<li><b>السونار:</b> بنقص منطقة الفحص نفسها، ونشيل المسطرة الجانبية ومربع المقاسات وبيانات الجهاز. العلامات اللي جوه منطقة الفحص (زي نقط القياس) بتفضل، ودي من حدود النموذج.</li>
<li>الصورة رمادي، فبنكررها 3 مرات علشان ResNet50 متعودة على صور ملونة (3 قنوات).</li>
</ul>
</div>

In [ ]:
from PIL import Image
from scipy import ndimage

IMG_SIZE = 224


def to_gray_uint8(path):
    im = Image.open(path)
    if im.mode in ('I;16', 'I;16B', 'I;16L', 'I', 'F'):          # صور 16-bit
        a = np.asarray(im, dtype=np.float32)
        lo, hi = np.percentile(a, (0.5, 99.5))
        return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
    return np.asarray(im.convert('L'))


def square_resize(a, size=IMG_SIZE, align_left=False):
    h, w = a.shape
    s = max(h, w)
    canvas = np.zeros((s, s), dtype=np.uint8)
    top = (s - h) // 2
    left = 0 if align_left else (s - w) // 2
    canvas[top:top + h, left:left + w] = a
    return np.asarray(Image.fromarray(canvas).resize((size, size), Image.Resampling.BILINEAR))


def largest_region_box(a, thr, step, opening, ignore_edges):
    # أكبر منطقة فاتحة في الصورة (على نسخة مصغرة علشان السرعة)
    mask = a[::step, ::step] > thr
    if ignore_edges:                                  # خطوط الإطار على حواف الماموجرام
        mh, mw = max(1, mask.shape[0] // 50), max(1, mask.shape[1] // 50)
        mask[:mh] = False
        mask[-mh:] = False
        mask[:, :mw] = False
        mask[:, -mw:] = False
    mask = ndimage.binary_opening(mask, iterations=opening)
    labels, n = ndimage.label(mask)
    if n == 0:
        return None
    sizes = ndimage.sum(mask, labels, range(1, n + 1))
    ys, xs = np.where(labels == int(np.argmax(sizes)) + 1)
    return ys.min() * step, (ys.max() + 1) * step, xs.min() * step, (xs.max() + 1) * step


def prep_mammogram(path):
    a = to_gray_uint8(path)
    k = max(1, a.shape[1] // 20)
    left, right = a[:, :k], a[:, -k:]
    bg_is_left = left.std() < right.std()           # الخلفية = الجنب الهادي اللي مفيهوش نسيج
    bg_level = float(np.median(left if bg_is_left else right))
    if bg_level > 127:                               # خلفية بيضا = الصورة معكوسة
        a = 255 - a
        bg_level = 255 - bg_level
    if bg_is_left:                                   # جدار الصدر يبقى على الشمال في كل الصور
        a = a[:, ::-1]
    box = largest_region_box(a, thr=bg_level + 25, step=8, opening=2, ignore_edges=True)
    if box is not None:
        r0, r1, c0, c1 = box
        a = a[r0:r1, c0:c1]
    return square_resize(a, align_left=True)


def prep_ultrasound(path):
    a = to_gray_uint8(path)
    box = largest_region_box(a, thr=12, step=2, opening=3, ignore_edges=False)
    if box is not None:
        r0, r1, c0, c1 = box
        if (r1 - r0) * (c1 - c0) > 0.2 * a.size:      # لو المنطقة صغيرة قوي، سيبي الصورة زي ما هي
            a = a[r0:r1, c0:c1]
    return square_resize(a)


# عينة قبل وبعد التجهيز: ماموجرام يمين، وماموجرام شمال، وسونار (من الصور اللي على الدرايف)
shown = images[images['on_drive']]
mammo = shown[shown['modality'] == 'Mammogram']
samples = [mammo.loc[mammo['side'] == 'R', 'path'].iloc[0], mammo.loc[mammo['side'] == 'L', 'path'].iloc[0],
           shown.loc[shown['modality'] == 'Ultrasound', 'path'].iloc[0]]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, (path, prep, name) in enumerate(zip(samples, [prep_mammogram, prep_mammogram, prep_ultrasound],
                                             ['mammogram R', 'mammogram L', 'ultrasound'])):
    axes[0, col].imshow(to_gray_uint8(path), cmap='gray')
    axes[0, col].set_title(f'{name} (original)')
    axes[1, col].imshow(prep(path), cmap='gray')
    axes[1, col].set_title(f'{name} (prepared)')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

<div dir="rtl">
<h3>ب-3) استخراج الميزات بـ ResNet50</h3>
<p>بنستخدم ResNet50 المدربة على ImageNet من غير الطبقة الأخيرة، فكل صورة بتطلع 2048 رقم بيوصفوا شكلها (Transfer Learning). ده خط أساس (baseline) بسيط ومناسب للبيانات الصغيرة.</p>
<p>الميزات بتتحفظ على Drive في <code>maaki_outputs</code>، فلو شغّلتي الدفتر تاني مش هيعيد الاستخراج. والصور اللي مش على الدرايف ميزاتها جاهزة من دفتر Zenodo.</p>
</div>

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(items, **kwargs):
        return items

print('GPU:', tf.config.list_physical_devices('GPU') or 'مفيش، هيشتغل على CPU وهياخد وقت أطول')
cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(IMG_SIZE, IMG_SIZE, 3))
PREP = {'Mammogram': prep_mammogram, 'Ultrasound': prep_ultrasound}


def run_batch(batch):
    x = preprocess_input(np.stack(batch).astype('float32'))
    return cnn.predict(x, verbose=0)


def extract(part, prep_fn, batch_size=16):
    feats, kept, batch, idx = [], [], [], []
    for i, path in tqdm(list(zip(part.index, part['path']))):
        try:
            g = prep_fn(path)
        except Exception as e:
            print(f'⚠️ تخطّينا {os.path.basename(path)}: {e}')
            continue
        batch.append(np.repeat(g[..., None], 3, axis=-1))
        idx.append(i)
        if len(batch) == batch_size:
            feats.append(run_batch(batch))
            kept += idx
            batch, idx = [], []
    if batch:
        feats.append(run_batch(batch))
        kept += idx
    out = pd.DataFrame(np.vstack(feats))
    out.index = part.loc[kept, 'path'].values
    return out


CACHE = os.path.join(OUT_DIR, 'bcmid_resnet50_features_v2.pkl')     # v2 = بعد تصليح تجهيز الماموجرام
feat = pd.read_pickle(CACHE) if os.path.exists(CACHE) else pd.DataFrame()
todo = images[~images['path'].isin(feat.index)]
lost = todo[~todo['on_drive']]
if len(lost):
    print(f'⚠️ {len(lost)} صورة مش على الدرايف ومالهاش ميزات محفوظة، فمش هتدخل. شغّلي دفتر Zenodo الأول.')
todo = todo[todo['on_drive']]
print(f'صور محتاجة استخراج: {len(todo)} من {len(images)}')
if len(todo):
    parts = [extract(part, PREP[modality]) for modality, part in todo.groupby('modality')]
    feat = pd.concat([feat] + parts)
    feat.to_pickle(CACHE)
feat = feat.loc[feat.index.intersection(images['path'])]
print('مصفوفة الميزات:', feat.shape)

<div dir="rtl">
<h3>ب-4) من الصورة للحالة</h3>
<p>كل حالة فيها كذا صورة. بناخد متوسط ميزات صور الماموجرام للحالة، ومتوسط ميزات صور السونار للحالة، كل نوع لوحده.</p>
</div>

In [ ]:
def case_matrix(modality):
    part = images[(images['modality'] == modality) & images['path'].isin(feat.index)]
    F = feat.loc[part['path']].copy()
    F.index = part['case'].values
    return F.groupby(level=0).mean()


FM_whole = case_matrix('Mammogram')      # الماموجرام بالصورة كلها (224×224)
FU = case_matrix('Ultrasound')
cases = ann.set_index('case')
cases = cases[cases.index.isin(FM_whole.index) | cases.index.isin(FU.index)]
print(f'حالات ليها تصنيف وصور: {len(cases)} (Positive = {int(cases["y"].sum())})')
print(f'منهم فيها ماموجرام: {cases.index.isin(FM_whole.index).sum()}، وفيها سونار: {cases.index.isin(FU.index).sum()}، '
      f'وفيها الاتنين: {(cases.index.isin(FM_whole.index) & cases.index.isin(FU.index)).sum()}')

<div dir="rtl">
<h3>ب-4-ب) الماموجرام بالمربعات</h3>
<p>صورة الماموجرام عرضها حوالي 5000 بكسل، ولما بنصغّرها لـ 224 (أصغر بحوالي 20 مرة) الورم الصغير والتكلسات بيختفوا. وكمان لما بناخد متوسط كل صور الحالة (الثديين والزاويتين)، إشارة صورة واحدة فيها ورم بتضيع وسط الباقي.</p>
<p>الحل اللي جربناه:</p>
<ol>
<li>نجهّز الماموجرام بنفس الطريقة، بس بمقاس 672×672 (3 أضعاف).</li>
<li>نقسم الصورة لـ 9 مربعات، كل مربع 224×224، ونسيب المربعات اللي فيها نسيج بس.</li>
<li>ResNet50 تطلّع ميزات كل مربع، وناخد أقوى قيمة لكل ميزة من المربعات كلها (max).</li>
<li>للحالة: أقوى قيمة من كل صورها، علشان لو صورة واحدة فيها حاجة تبان.</li>
</ol>
<p>الفكرة دي اسمها Multiple Instance Learning: الحالة «Positive» لو أي جزء منها فيه حاجة. الخلية دي بتاخد حوالي 10 دقايق أول مرة، وبعد كده بتتحفظ.</p>
</div>

In [ ]:
GRID = 3
BIG = IMG_SIZE * GRID                     # 672 × 672 بدل 224 × 224


def prep_mammogram_big(path):
    # نفس تجهيز الماموجرام بالظبط، بس بمقاس أكبر
    a = to_gray_uint8(path)
    k = max(1, a.shape[1] // 20)
    left, right = a[:, :k], a[:, -k:]
    bg_is_left = left.std() < right.std()
    bg_level = float(np.median(left if bg_is_left else right))
    if bg_level > 127:
        a = 255 - a
        bg_level = 255 - bg_level
    if bg_is_left:
        a = a[:, ::-1]
    box = largest_region_box(a, thr=bg_level + 25, step=8, opening=2, ignore_edges=True)
    if box is not None:
        r0, r1, c0, c1 = box
        a = a[r0:r1, c0:c1]
    return square_resize(a, size=BIG, align_left=True)


def tile_feature(path):
    g = prep_mammogram_big(path)
    tiles = [g[r:r + IMG_SIZE, c:c + IMG_SIZE] for r in range(0, BIG, IMG_SIZE) for c in range(0, BIG, IMG_SIZE)]
    tiles = [t for t in tiles if (t > 40).mean() > 0.2] or tiles        # المربعات اللي فيها نسيج بس
    x = preprocess_input(np.stack([np.repeat(t[..., None], 3, axis=-1) for t in tiles]).astype('float32'))
    return cnn.predict(x, verbose=0).max(axis=0)                         # أقوى إشارة في أي مربع


TILE_CACHE = os.path.join(OUT_DIR, 'bcmid_mammo_tiles_v1.pkl')
tile_feat = pd.read_pickle(TILE_CACHE) if os.path.exists(TILE_CACHE) else pd.DataFrame()
todo = images[(images['modality'] == 'Mammogram') & ~images['path'].isin(tile_feat.index)]
lost = todo[~todo['on_drive']]
if len(lost):
    print(f'⚠️ {len(lost)} صورة ماموجرام مش على الدرايف ومالهاش ميزات مربعات محفوظة، فمش هتدخل. شغّلي دفتر Zenodo الأول.')
todo = todo[todo['on_drive']]
print(f'صور ماموجرام محتاجة استخراج بالمربعات: {len(todo)}')
new_rows = {}
for path in tqdm(list(todo['path'])):
    try:
        new_rows[path] = tile_feature(path)
    except Exception as e:
        print(f'⚠️ تخطّينا {os.path.basename(path)}: {e}')
if new_rows:
    tile_feat = pd.concat([tile_feat, pd.DataFrame.from_dict(new_rows, orient='index')])
    tile_feat.to_pickle(TILE_CACHE)

part = images[(images['modality'] == 'Mammogram') & images['path'].isin(tile_feat.index)]
FT = tile_feat.loc[part['path']].copy()
FT.index = part['case'].values
FM_tiles = FT.groupby(level=0).max()      # للحالة: أقوى إشارة في أي صورة
print('مصفوفة الماموجرام بالمربعات (حالات × ميزات):', FM_tiles.shape)

<div dir="rtl">

<h3>ب-5) طريقة التقييم</h3>
<ul>
<li><b>التقسيم حسب المريضة:</b> <code>StratifiedGroupKFold</code> بيحط كل حالات المريضة الواحدة في نفس الجزء، فمستحيل حالة ليها تكون في التدريب وحالة تانية ليها في الاختبار.</li>
<li><b>5 أجزاء × 3 تقسيمات مختلفة = 15 تقييم.</b></li>
<li><b>النموذج:</b> StandardScaler ← PCA (50 مكون) ← Logistic Regression. الـ PCA بيختصر الـ 2048 رقم لـ 50 لأن عدد الحالات قليل، وده بيقلل الـ overfitting. و<code>class_weight='balanced'</code> علشان الحالات الـ Positive أقل بكتير.</li>
<li><b>الدمج (Late Fusion):</b> في كل fold بندرّب نموذج ماموجرام ونموذج سونار، وبناخد متوسط الاحتمالين للحالة. لو الحالة فيها نوع واحد بس، بناخد احتماله. الدمج هنا صح لأن الصورتين لنفس المريضة.</li>
<li><b>الماموجرام بطريقتين:</b> الأول بالصورة كلها (ب-4)، وبعدين بالمربعات (ب-4-ب). بنعتمد طريقة المربعات. وبما إننا اخترناها بعد ما شفنا نتيجة الاتنين، رقمها ممكن يكون متفائل شوية.</li>
</ul>

</div>

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, accuracy_score


def make_head(n_train):
    return Pipeline([('scale', StandardScaler()),
                     ('pca', PCA(n_components=min(50, n_train - 1), random_state=SEED)),
                     ('clf', LogisticRegression(C=0.1, class_weight='balanced', max_iter=5000))])


def evaluate(cases, FM, FU, seeds=(0, 1, 2), n_splits=5):
    rows, oof = [], []
    for seed in seeds:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for fold, (tr, te) in enumerate(splitter.split(cases, cases['y'], cases['patient'])):
            tr_cases, te_cases = cases.index[tr], cases.index[te]
            probs = {}
            for name, F in [('Mammogram', FM), ('Ultrasound', FU)]:
                tr_ok = [c for c in tr_cases if c in F.index]
                te_ok = [c for c in te_cases if c in F.index]
                head = make_head(len(tr_ok)).fit(F.loc[tr_ok], cases.loc[tr_ok, 'y'])
                probs[name] = pd.Series(head.predict_proba(F.loc[te_ok])[:, 1], index=te_ok)
            probs['Fusion'] = pd.concat(probs, axis=1).mean(axis=1)      # متوسط الاحتمالين المتاحين
            for name, p in probs.items():
                y_true = cases.loc[p.index, 'y']
                pred = (p >= 0.5).astype(int)
                rows.append({'seed': seed, 'fold': fold, 'model': name, 'n_test': len(p),
                             'AUC': roc_auc_score(y_true, p) if y_true.nunique() == 2 else np.nan,
                             'Sensitivity': recall_score(y_true, pred, pos_label=1, zero_division=0),
                             'Specificity': recall_score(y_true, pred, pos_label=0, zero_division=0),
                             'Balanced acc.': balanced_accuracy_score(y_true, pred),
                             'Accuracy': accuracy_score(y_true, pred)})
                oof.append(pd.DataFrame({'case': p.index, 'prob': p.values, 'y': y_true.values,
                                         'model': name, 'seed': seed}))
    return pd.DataFrame(rows), pd.concat(oof, ignore_index=True)


def summarize(folds):
    metrics = ['AUC', 'Sensitivity', 'Specificity', 'Balanced acc.', 'Accuracy']
    g = folds.groupby('model')[metrics]
    mean, sd = g.mean(), g.std()
    table = mean.map('{:.2f}'.format) + ' ± ' + sd.map('{:.2f}'.format)
    return table.loc[['Mammogram', 'Ultrasound', 'Fusion']], mean, sd


whole_folds, _ = evaluate(cases, FM_whole, FU)
whole_table, whole_mean, whole_sd = summarize(whole_folds)
img_folds, img_oof = evaluate(cases, FM_tiles, FU)
img_table, img_mean, img_sd = summarize(img_folds)

print('1) الماموجرام بالصورة كلها (224×224):')
display(whole_table.loc[['Mammogram']])
print('\n2) الماموجرام بالمربعات، والسونار، والدمج (دي النتيجة اللي بنعتمدها):')
display(img_table)

<div dir="rtl">
<h3>ب-6) منحنى ROC ومصفوفة الالتباس</h3>
<p>من أول تقسيمة (seed 0): كل حالة خدت احتمال من نموذج ما شافهاش. مصفوفة الالتباس للدمج عند حد 0.5.</p>
</div>

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for name in ['Mammogram', 'Ultrasound', 'Fusion']:
    o = img_oof[(img_oof['model'] == name) & (img_oof['seed'] == 0)]
    fpr, tpr, _ = roc_curve(o['y'], o['prob'])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(o["y"], o["prob"]):.2f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='random')
axes[0].set_xlabel('1 - specificity')
axes[0].set_ylabel('sensitivity')
axes[0].legend()
o = img_oof[(img_oof['model'] == 'Fusion') & (img_oof['seed'] == 0)]
ConfusionMatrixDisplay(confusion_matrix(o['y'], (o['prob'] >= 0.5).astype(int)),
                       display_labels=['BI-RADS 1-3', 'BI-RADS 4-6']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Fusion, threshold 0.5')
plt.tight_layout()
plt.show()

<div dir="rtl">
<h3>ب-7) اختبار إضافي: من غير BI-RADS 6</h3>
<p>BI-RADS 6 معناها سرطان متأكد قبل كده بالعيّنة، وغالبًا الصورة فيها علامات علاج أو ورم واضح، فده بيسهّل الشغل على النموذج. بنعيد التقييم من غيرهم علشان نشوف النموذج لسه بيفرّق ولا لأ.</p>
</div>

In [ ]:
strict = cases[cases['birads'] != 6]
strict_folds, _ = evaluate(strict, FM_tiles, FU)
strict_table, strict_mean, strict_sd = summarize(strict_folds)
print(f'من غير BI-RADS 6: {len(strict)} حالة (Positive = {int(strict["y"].sum())})')
display(strict_table)

<div dir="rtl">
<h3>ب-7-ب) هل النموذج بيعتمد على اختصار؟</h3>
<p>أحيانًا النموذج بيلاقي «اختصار» (shortcut): معلومة مرتبطة بالتصنيف ومالهاش علاقة بشكل النسيج، فنتيجته تطلع عالية، بس مش للسبب الصح. بنعمل 4 اختبارات، وطريقة قراية كل اختبار مكتوبة قبل ما نشوف أرقامه:</p>
<ol>
<li><b>السنة:</b> في BCMID، نسبة الـ Positive في الحالات من 2020 وبعدها أعلى بكتير من اللي قبلها (الجدول الأول تحت). فلو الصورة بتبيّن السنة بأي شكل، زي جهاز مختلف، النموذج ممكن يتعلم «الحالة جديدة، يبقى غالبًا Positive» من غير ما يبص على الورم. علشان نختبر ده، بنحسب AUC تاني بيقارن كل حالة Positive بالحالات الـ Negative اللي من <b>نفس السنة</b> بس.<br>
<b>القراية:</b> لو الرقم ده قريب من الـ AUC العادي، يبقى السنة مش هي اللي شايلة النتيجة. ولو نزل قريب من 0.5، يبقى معظم الأداء جاي من السنة. والرقم ده بيتذبذب أكتر، لأن عدد المقارنات أقل.</li>
<li><b>نموذج مابيشوفش الصور خالص:</b> بياخد معلومات جانبية بس عن كل حالة: عدد صور الماموجرام، وعدد الصور الإضافية منها، ومتوسط حجم ملف الماموجرام (بيختلف من جهاز للتاني)، وعدد صور السونار، والسنة. بنقيّمه بنفس التقسيم حسب المريضة، بـ Logistic Regression وRandom Forest.<br>
<b>القراية:</b> ده الرقم اللي ممكن نوصله من غير ما نبص في الصور. وكل ما نموذج الصور يبقى أعلى منه بفرق أكبر، كل ما نطمن إن الصور بتضيف معلومة حقيقية.</li>
<li><b>الماموجرام بأول صورة بس من كل زاوية:</b> فيه حالات ليها أكتر من صورة لنفس الثدي ونفس الزاوية (اسم الملف بيخلص بـ _2 أو _3). الصور الزيادة دي ممكن تكون صور إضافية الطبيب طلبها علشان شك في حاجة، وطريقة المربعات بتاخد أقوى إشارة من كل صور الحالة، فممكن تعتمد عليها.<br>
<b>القراية:</b> لو الأداء فضل قريب من غيرها، يبقى الصور الإضافية مش هي اللي شايلة النتيجة. ولو نزل كتير، فده مش غش بالظبط، لأن الصور دي فيها معلومة حقيقية عن الثدي. بس معناه إن النموذج محتاج صور مش هتبقى موجودة في أول كشف.</li>
<li><b>نفس عدد الصور ونفس الفترة</b> (الاختبار ده ضفناه بعد ما شفنا نتيجة التلاتة اللي فاتوا): عدد صور الماموجرام في الحالة طلع أقوى معلومة جانبية لوحده. وطريقة المربعات بتاخد أقوى إشارة من كل صور الحالة، فالحالة اللي ليها صور أكتر ممكن تطلع إشارتها مختلفة بس علشان عدد صورها. فبنحسب الـ AUC بين حالات ليها <b>نفس عدد صور الماموجرام ومن نفس الفترة</b> بس، لنماذج الصور ولنموذج المعلومات الجانبية.<br>
<b>القراية:</b> لو نموذج الصور فضل أعلى بوضوح من 0.5 ومن نموذج المعلومات الجانبية جوه المجموعات دي، يبقى الصور فيها معلومة حقيقية زيادة على تركيبة الحالة. ولو قرب من 0.5، يبقى معظم الأداء جاي من تركيبة الحالة مش من الصورة.</li>
</ol>
</div>

In [ ]:
from sklearn.metrics import roc_auc_score

yb = cases['y']
year = pd.Series([int(re.search(r'y(\d\d)', c).group(1)) for c in cases.index], index=cases.index)   # أول رقمين بعد y
period = pd.Series(np.where(year >= 20, 'من 2020', 'قبل 2020'), index=cases.index)
mam_imgs = images[images['modality'] == 'Mammogram'].copy()
mam_imgs['num'] = mam_imgs['file'].str.extract(r'_(\d+)\.\w+$')[0].astype(float).fillna(1)         # _1 أو _2 أو _3
us_imgs = images[images['modality'] == 'Ultrasound']
mam_mb = (mam_imgs['bytes'] / 1e6).groupby(mam_imgs['case']).mean().reindex(cases.index)

print('الحالات حسب الفترة:')
display(pd.DataFrame({'عدد الحالات': period.value_counts(),
                      'Positive': yb.groupby(period).sum(),
                      'نسبة الـ Positive': yb.groupby(period).mean().round(2),
                      'متوسط حجم ملف الماموجرام (MB)': mam_mb.groupby(period).mean().round(1)}))

# المعلومات الجانبية عن كل حالة، من غير ما نبص في الصور
side_info = pd.DataFrame({
    'عدد صور الماموجرام': mam_imgs.groupby('case').size(),
    'عدد صور الماموجرام الإضافية': (mam_imgs['num'] > 1).groupby(mam_imgs['case']).sum(),
    'متوسط حجم ملف الماموجرام (MB)': mam_mb,
    'عدد صور السونار': us_imgs.groupby('case').size(),
    'السنة': year,
}).reindex(cases.index).fillna(0)

single = pd.Series({col: roc_auc_score(yb, side_info[col]) for col in side_info.columns})
single = single.where(single >= 0.5, 1 - single)        # الاتجاه مش مهم: 0.5 = مالهاش علاقة بالتصنيف
print('\nكل معلومة جانبية لوحدها (AUC من غير تدريب، و0.5 = مالهاش علاقة بالتصنيف):')
display(single.round(2).to_frame('AUC'))


def side_info_cv(make_model, seeds=(0, 1, 2), n_splits=5):
    # نفس التقسيم حسب المريضة اللي في ب-5، وبنحتفظ بتوقع كل حالة من نموذج ما شافهاش
    aucs, oof = [], []
    for seed in seeds:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, te in splitter.split(side_info, yb, cases['patient']):
            model = make_model().fit(side_info.iloc[tr], yb.iloc[tr])
            p = model.predict_proba(side_info.iloc[te])[:, 1]
            aucs.append(roc_auc_score(yb.iloc[te], p))
            oof.append(pd.DataFrame({'case': side_info.index[te], 'prob': p, 'y': yb.iloc[te].values, 'seed': seed}))
    return (np.mean(aucs), np.std(aucs, ddof=1)), pd.concat(oof, ignore_index=True)


side_lr, lr_oof = side_info_cv(lambda: Pipeline([('scale', StandardScaler()),
                                                 ('clf', LogisticRegression(class_weight='balanced', max_iter=5000))]))
side_rf, rf_oof = side_info_cv(lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                              class_weight='balanced', random_state=SEED))

# الماموجرام بالمربعات، بأول صورة بس من كل زاوية
extra = side_info['عدد صور الماموجرام الإضافية'] > 0
print(f"\nصور ماموجرام إضافية (_2 أو أكتر): {int((mam_imgs['num'] > 1).sum())} من {len(mam_imgs)}، في {int(extra.sum())} حالة")
if extra.any():
    print(f"نسبة الـ Positive: {yb[extra].mean():.2f} في الحالات اللي ليها صور إضافية، و{yb[~extra].mean():.2f} في الباقي")
first = mam_imgs[(mam_imgs['num'] == 1) & mam_imgs['path'].isin(tile_feat.index)]
FT1 = tile_feat.loc[first['path']].copy()
FT1.index = first['case'].values
FM_first = FT1.groupby(level=0).max()
first_folds, _ = evaluate(cases, FM_first, FU)
first_table, first_mean, first_sd = summarize(first_folds)


def ms(mean, sd):
    return f'{mean:.2f} ± {sd:.2f}'


check = pd.DataFrame({'AUC': [
    ms(img_mean.loc['Fusion', 'AUC'], img_sd.loc['Fusion', 'AUC']),
    ms(img_mean.loc['Mammogram', 'AUC'], img_sd.loc['Mammogram', 'AUC']),
    ms(first_mean.loc['Mammogram', 'AUC'], first_sd.loc['Mammogram', 'AUC']),
    ms(*side_lr),
    ms(*side_rf)]},
    index=['بالصور: الدمج (النتيجة اللي بنعتمدها)',
           'بالصور: الماموجرام بالمربعات، كل الصور',
           'بالصور: الماموجرام بالمربعات، أول صورة بس من كل زاوية',
           'من غير صور: معلومات جانبية (Logistic Regression)',
           'من غير صور: معلومات جانبية (Random Forest)'])
print('\nالمقارنة:')
display(check)
best_side = max(side_lr[0], side_rf[0])
print(f"\nالدمج بالصور أعلى من أحسن نموذج من غير صور بـ {img_mean.loc['Fusion', 'AUC'] - best_side:+.2f} في الـ AUC")
print(f"الماموجرام اتغيّر بـ {first_mean.loc['Mammogram', 'AUC'] - img_mean.loc['Mammogram', 'AUC']:+.2f} "
      f"في الـ AUC لما شلنا الصور الإضافية")


def within_auc(prob, y_true, groups):
    # بنقارن كل حالة Positive بالحالات الـ Negative اللي في نفس مجموعتها بس
    hits = pairs = 0
    for _, g in pd.DataFrame({'p': prob, 'y': y_true, 'g': groups}).groupby('g'):
        diff = g.loc[g['y'] == 1, 'p'].values[:, None] - g.loc[g['y'] == 0, 'p'].values[None, :]
        hits += (diff > 0).sum() + 0.5 * (diff == 0).sum()
        pairs += diff.size
    return (hits / pairs if pairs else np.nan), pairs


GROUPS = {'نفس السنة': year,
          'نفس عدد صور الماموجرام ونفس الفترة': side_info['عدد صور الماموجرام'].astype(int).astype(str) + ' | ' + period}
OOFS = {'بالصور: Mammogram': img_oof[img_oof['model'] == 'Mammogram'],
        'بالصور: Ultrasound': img_oof[img_oof['model'] == 'Ultrasound'],
        'بالصور: Fusion': img_oof[img_oof['model'] == 'Fusion'],
        'من غير صور: Random Forest': rf_oof}
rows = []
for name, o in OOFS.items():
    for seed, part in o.groupby('seed'):
        row = {'النموذج': name, 'كل الحالات': roc_auc_score(part['y'], part['prob'])}
        for gname, g in GROUPS.items():
            row[gname] = within_auc(part['prob'].values, part['y'].values, g.loc[part['case']].values)[0]
        rows.append(row)
within = pd.DataFrame(rows).groupby('النموذج', sort=False).mean().round(2)
print('\nالـ AUC على كل الحالات، وجوه المجموعات بس (من نفس التوقعات):')
display(within)
all_pairs = int(yb.sum() * (1 - yb).sum())
for gname, g in GROUPS.items():
    print(f'المقارنات جوه «{gname}»: {within_auc(np.zeros(len(yb)), yb.values, g.values)[1]} من {all_pairs}')

<div dir="rtl">
<h4>الخلاصة من ب-7-ب</h4>
<ul>
<li><b>السنة:</b> الأداء نزل شوية بس لما قارنّا حالات من نفس السنة، فالسنة لوحدها مش هي اللي شايلة النتيجة.</li>
<li><b>الصور الإضافية:</b> الحالات اللي ليها صور إضافية نسبة الـ Positive فيها أقل مش أعلى، والأداء نزل شوية بس من غيرها. فهي كمان مش اللي شايلة النتيجة.</li>
<li><b>المعلومات الجانبية:</b> على كل الحالات، نموذج مابيشوفش الصور خالص جاب AUC قد نموذج الصور أو أعلى منه شوية، وأقوى معلومة لوحدها كانت عدد صور الماموجرام في الحالة. يعني تصنيف BCMID مرتبط جامد بتركيبة الحالة: اتصورت كام صورة، وفي أنهي سنة.</li>
<li><b>نفس عدد الصور ونفس الفترة:</b> لما قارنّا حالات متشابهة في الحاجتين دول، النموذج اللي من غير صور نزل لقريب من 0.5، يعني الاختصارات اتشالت فعلًا. السونار فضل زي ما هو تقريبًا، والدمج فضل أعلى بوضوح من النموذج اللي من غير صور. أما الماموجرام فنزل كتير، يعني معظم قوته الظاهرة كانت جاية من عدد الصور، لأن طريقة المربعات بتاخد أقوى إشارة من كل صور الحالة.</li>
<li><b>النتيجة:</b> الصور فيها معلومة حقيقية زيادة على تركيبة الحالة، وأغلبها جاي من السونار. ورقم الماموجرام على كل الحالات متفائل جدًا، والأصدق ليه هو الرقم اللي جوه المجموعات. والاختبار على داتا من مستشفى تانية (CDD-CESM) هو اللي هيأكد ده.</li>
</ul>
</div>

<div dir="rtl">
<h3>ب-8) مقارنة بالأبحاث المنشورة على نفس الداتا</h3>
<p>ورقة «Knowing When to Defer» (مجلة BioMedInformatics من MDPI، 2026) استخدمت BCMID بنماذج متخصصة في الثدي وتقنية LoRA، وقسمت الحالات 3 فئات. أحسن نموذج عندهم (ماموجرام + سونار) جاب AUC للفئة الـ Positive حوالي 0.87، ومتوسط AUC للتلات فئات حوالي 0.76. ونموذج CNN عادي متدرب على ImageNet عندهم جاب حوالي 0.66 للتلات فئات.</p>
<p>المقارنة تقريبية ومش عادلة، لأن احنا عاملين فئتين بس (BI-RADS 3 مع الـ Negative)، وهما عاملين تلات فئات. ولو رقمنا طلع قريب من نماذجهم المتخصصة، ده مش معناه إن نموذجنا البسيط أحسن منهم: اخترنا طريقة المربعات بعد ما شفنا النتيجة، والداتا صغيرة ومن مستشفى واحد. وكمان في ب-7-ب لقينا إن نموذج مابيشوفش الصور خالص بيوصل لـ AUC عالي على BCMID، فأي رقم على الداتا دي، بتاعنا أو منشور، محتاج الاختبار ده قبل ما نعتمد عليه.</p>
</div>

<div dir="rtl">
<h3>ب-9) النماذج النهائية وحفظها</h3>
<p>بندرّب نموذج الماموجرام ونموذج السونار على كل الحالات، ونحفظهم. علشان التطبيق يستخدمهم، السيرفر لازم يعمل نفس تجهيز الصور ونفس ResNet50 الأول، وبعدين يدي الميزات للنموذج.</p>
</div>

In [ ]:
heads = {}
for name, F in [('Mammogram', FM_tiles), ('Ultrasound', FU)]:
    ok = [c for c in cases.index if c in F.index]
    heads[name] = make_head(len(ok)).fit(F.loc[ok], cases.loc[ok, 'y'])
    joblib.dump(heads[name], os.path.join(OUT_DIR, f'bcmid_{name.lower()}_head.joblib'))

img_meta = {
    'task': 'BI-RADS 4-6 (positive) vs BI-RADS 1-3 (negative); BI-RADS 0 excluded',
    'labels': 'radiologist BI-RADS, not biopsy-confirmed',
    'feature_extractor': 'ResNet50 (ImageNet weights), include_top=False, pooling=avg, input 224x224 gray repeated to 3 channels',
    'preprocessing': {'Mammogram': 'detect background side; invert if bright; flip so chest wall is on the left; crop largest '
                                   'bright region; pad to square (left aligned); resize 672; 3x3 tiles of 224 that contain tissue',
                      'Ultrasound': 'crop scan area (largest bright region), pad to square, resize 224'},
    'case_level': 'Mammogram: max of tile features, then max over the images of the case. Ultrasound: mean of image '
                  'features. Fusion = mean of the two probabilities',
    'evaluation': 'StratifiedGroupKFold by patient, 5 folds x 3 seeds',
    'side_info_only_auc': [round(float(side_rf[0]), 3), round(float(side_rf[1]), 3)],
    'within_group_auc': {g: {m: float(v) for m, v in within[g].items()} for g in within.columns},
    'warning': 'On BCMID, a model that sees no images (number of images per case, year, file size) reaches about the '
               'same AUC as the image models over all cases. Within groups with the same number of mammogram images and '
               'the same period (see within_group_auc), ultrasound and fusion keep their AUC but the mammogram drops a '
               'lot: its case score (max over images) depends on how many images the case has. External validation on '
               'another hospital is required before any claim.',
    'cv_metrics': {m: {k: [round(float(img_mean.loc[m, k]), 3), round(float(img_sd.loc[m, k]), 3)] for k in img_mean.columns}
                   for m in img_mean.index},
    'data': 'BCMID, Ayady Almostakbal Hospital, Alexandria (CC BY 4.0, DOI 10.5281/zenodo.19402690)',
    'images_used': {'Mammogram': int(images.loc[images['modality'] == 'Mammogram', 'path'].isin(tile_feat.index).sum()),
                    'Ultrasound': int(images.loc[images['modality'] == 'Ultrasound', 'path'].isin(feat.index).sum())},
    'image_list': 'official file list of the Zenodo zip (bcmid_manifest.csv) when present; features of images not stored '
                  'on Drive were computed from the zip with the same code (maaki_bcmid_zenodo_features.ipynb)',
    'trained_on': str(date.today()),
}
with open(os.path.join(OUT_DIR, 'bcmid_model.json'), 'w', encoding='utf-8') as f:
    json.dump(img_meta, f, ensure_ascii=False, indent=2)
print('اتحفظ في:', OUT_DIR)

<div dir="rtl">
<h3>ب-9-ب) النسخة اللي بتدخل التطبيق</h3>
<p>في ب-7-ب لقينا إن الماموجرام بياخد أقوى إشارة من كل صور الحالة، فنتيجته بتتأثر بعدد الصور. وفي التطبيق المريضة ممكن ترفع صورة واحدة أو أربعة، فمينفعش النتيجة تتغير بسبب العدد. علشان كده النسخة اللي بتدخل التطبيق:</p>
<ul>
<li><b>الماموجرام:</b> كل صورة بتتقسم مربعات زي ما هي، بس الحالة بتاخد <b>متوسط</b> صورها بدل أقواها.</li>
<li><b>السونار:</b> زي ما هو، متوسط الصور.</li>
<li><b>الدمج:</b> متوسط الاحتمالين المتاحين.</li>
</ul>
<p>بنقيّمها بنفس التقسيم ونفس اختبار المجموعات اللي في ب-7-ب. وبعدين بنصدّر النموذج كأرقام (ملف <code>npz</code>) مش كملف pickle، علشان السيرفر يحسبه بـ numpy بس، ونتأكد إن الحسبة دي هي هي حسبة sklearn. وكمان بنسجّل حالة اختبار: صور معيّنة والنتيجة المتوقعة ليها، علشان نتأكد إن السيرفر بيطلّع نفس الأرقام. وفي الآخر بيتنزل ملف <code>maaki_app_models.zip</code> على جهازك.</p>
</div>

In [ ]:
import shutil
import sklearn

# 1) الماموجرام بمتوسط الصور بدل أقواها، والتقييم بنفس الطريقة
FM_app = FT.groupby(level=0).mean()
app_folds, app_oof = evaluate(cases, FM_app, FU)
app_table, app_mean, app_sd = summarize(app_folds)
rows = []
for name in ['Mammogram', 'Ultrasound', 'Fusion']:
    for seed, part in app_oof[app_oof['model'] == name].groupby('seed'):
        row = {'النموذج': name, 'كل الحالات': roc_auc_score(part['y'], part['prob'])}
        for gname, g in GROUPS.items():
            row[gname] = within_auc(part['prob'].values, part['y'].values, g.loc[part['case']].values)[0]
        rows.append(row)
app_within = pd.DataFrame(rows).groupby('النموذج', sort=False).mean().round(2)
print('نسخة التطبيق (الماموجرام بمتوسط الصور):')
display(app_table)
print('\nوجوه المجموعات (زي ب-7-ب):')
display(app_within)

# 2) التدريب على كل الحالات، والتصدير كأرقام
app_heads = {}
for name, F in [('Mammogram', FM_app), ('Ultrasound', FU)]:
    ok = [c for c in cases.index if c in F.index]
    app_heads[name] = make_head(len(ok)).fit(F.loc[ok], cases.loc[ok, 'y'])

arrays = {}
for name, head in app_heads.items():
    k = name.lower()
    arrays[f'{k}_scale_mean'] = head['scale'].mean_
    arrays[f'{k}_scale_std'] = head['scale'].scale_
    arrays[f'{k}_pca_mean'] = head['pca'].mean_
    arrays[f'{k}_pca_components'] = head['pca'].components_
    arrays[f'{k}_coef'] = head['clf'].coef_[0]
    arrays[f'{k}_intercept'] = head['clf'].intercept_


def head_prob(x, k):
    # نفس حساب النموذج بالأرقام بس، زي ما السيرفر هيعمله
    z = (np.asarray(x, dtype=np.float64) - arrays[f'{k}_scale_mean']) / arrays[f'{k}_scale_std']
    p = (z - arrays[f'{k}_pca_mean']) @ arrays[f'{k}_pca_components'].T
    return 1 / (1 + np.exp(-(p @ arrays[f'{k}_coef'] + arrays[f'{k}_intercept'][0])))


print()
for name, F in [('Mammogram', FM_app), ('Ultrasound', FU)]:
    ok = [c for c in cases.index if c in F.index]
    diff = np.abs(head_prob(F.loc[ok].values, name.lower()) - app_heads[name].predict_proba(F.loc[ok])[:, 1]).max()
    print(f'{name}: أكبر فرق بين الحسبة بالأرقام وحسبة sklearn = {diff:.1e}')

# 3) حالة اختبار: السيرفر لازم يطلّع نفس الأرقام دي لنفس الصور
on = images[images['on_drive']]
both = [c for c in cases.index if {'Mammogram', 'Ultrasound'} <= set(on.loc[on['case'] == c, 'modality'])]
test_case = both[0]
tm = on[(on['case'] == test_case) & (on['modality'] == 'Mammogram') & on['path'].isin(tile_feat.index)]
tu = on[(on['case'] == test_case) & (on['modality'] == 'Ultrasound') & on['path'].isin(feat.index)]
p_m = float(head_prob(tile_feat.loc[tm['path']].values.mean(axis=0, keepdims=True), 'mammogram')[0])
p_u = float(head_prob(feat.loc[tu['path']].values.mean(axis=0, keepdims=True), 'ultrasound')[0])
self_test = {'case': test_case,
             'mammogram_files': [f'{test_case}/Mammogram/{f}' for f in tm['file']],
             'ultrasound_files': [f'{test_case}/Ultrasound/{f}' for f in tu['file']],
             'expected': {'Mammogram': round(p_m, 4), 'Ultrasound': round(p_u, 4), 'Fusion': round((p_m + p_u) / 2, 4)}}
print('\nحالة الاختبار:', json.dumps(self_test, ensure_ascii=False))


def stats(mean_df, sd_df):
    return {m: {k: [round(float(mean_df.loc[m, k]), 3), round(float(sd_df.loc[m, k]), 3)] for k in mean_df.columns}
            for m in mean_df.index}


app_card = {
    'name': 'Maaki BCMID image model (app version)',
    'predicts': 'probability that a radiologist would rate the case BI-RADS 4-6 (suspicious, needs biopsy, or known '
                'cancer) rather than BI-RADS 1-3',
    'not_a_diagnosis': 'Decision support for the doctor only. Labels are radiologist BI-RADS, not biopsy results.',
    'data': 'BCMID v3, Ayady Almostakbal Hospital, Alexandria (CC BY 4.0, DOI 10.5281/zenodo.19402690)',
    'cases': int(len(cases)),
    'positives': int(cases['y'].sum()),
    'feature_extractor': {'model': 'tf.keras.applications.ResNet50', 'weights': 'imagenet', 'include_top': False,
                          'pooling': 'avg', 'input': [IMG_SIZE, IMG_SIZE, 3],
                          'preprocess_input': 'tf.keras.applications.resnet50.preprocess_input',
                          'gray_to_rgb': 'repeat the gray channel 3 times'},
    'mammogram': {'image': 'prep_mammogram_big (672x672), 3x3 tiles of 224, keep tiles with tissue, max over tiles',
                  'case': 'mean over the case images'},
    'ultrasound': {'image': 'prep_ultrasound (224x224)', 'case': 'mean over the case images'},
    'fusion': 'mean of the available probabilities',
    'threshold': 0.5,
    'cv_metrics': stats(app_mean, app_sd),
    'within_group_auc': {key: {m: float(v) for m, v in app_within[col].items()} for key, col in
                         [('all_cases', 'كل الحالات'), ('same_year', 'نفس السنة'),
                          ('same_image_count_and_period', 'نفس عدد صور الماموجرام ونفس الفترة')]},
    'side_info_only_auc': [round(float(side_rf[0]), 3), round(float(side_rf[1]), 3)],
    'limitations': ['single hospital (Alexandria) and a small dataset',
                    'labels are radiologist BI-RADS assessments, not biopsy results',
                    'on BCMID, case structure (number of images, year) predicts the label; see within_group_auc',
                    'ultrasound images contain caliper marks placed by the radiologist',
                    'not yet tested on data from another hospital'],
    'self_test': self_test,
    'versions': {'tensorflow': tf.__version__, 'scikit-learn': sklearn.__version__, 'numpy': np.__version__},
    'trained_on': str(date.today()),
}

APP_DIR = os.path.join(OUT_DIR, 'maaki_app_models')
os.makedirs(APP_DIR, exist_ok=True)
np.savez(os.path.join(APP_DIR, 'bcmid_heads.npz'), **arrays)
with open(os.path.join(APP_DIR, 'bcmid_app_model.json'), 'w', encoding='utf-8') as f:
    json.dump(app_card, f, ensure_ascii=False, indent=2)
zip_path = shutil.make_archive('/content/maaki_app_models' if IN_COLAB else os.path.join(OUT_DIR, 'maaki_app_models'),
                               'zip', APP_DIR)
print('\nاتحفظ في:', APP_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)           # الملف بيتنزل على جهازك في Downloads

<div dir="rtl">
<h3>ب-10) حدود نموذج الصور (لازم تقوليها)</h3>
<ul>
<li>التصنيف تقييم BI-RADS من طبيب الأشعة، مش نتيجة عيّنة.</li>
<li>مستشفى واحد، وعدد حالات قليل. وحسب الورقة المنشورة، أغلب الحالات بعد جراحة (استئصال جزئي أو كامل)، فالنموذج ممكن يتعلم علامات الجراحة.</li>
<li>صور السونار جواها علامات القياس اللي بيحطها الطبيب لما يلاقي حاجة، فالنموذج ممكن يتعلم «في علامة قياس» بدل «في ورم» (shortcut learning).</li>
<li>حتى بالمربعات، الماموجرام بقى 672 بكسل بس من حوالي 5000، فالتكلسات الدقيقة ممكن لسه تضيع.</li>
<li><b>أهم حد:</b> تصنيف BCMID مرتبط جامد بتركيبة الحالة: عدد الصور اللي اتصورت، والسنة. على كل الحالات، نموذج مابيشوفش الصور خالص جاب AUC قد نموذج الصور تقريبًا (ب-7-ب). ولما قارنّا حالات بنفس عدد الصور ونفس الفترة، السونار والدمج فضلوا كويسين، لكن الماموجرام نزل كتير. فرقم الماموجرام في الملخص متفائل، ومحتاجين اختبار على داتا من مستشفى تانية.</li>
<li>نموذج الماموجرام في ب-5 بياخد أقوى إشارة من كل صور الحالة، فنتيجته بتتأثر بعدد الصور. علشان كده النسخة اللي بتدخل التطبيق (ب-9-ب) بتاخد متوسط الصور بدل أقواها، فالنتيجة ماتتغيرش لو المريضة رفعت صورتين بدل أربعة.</li>
<li>اختبارات الاختصارات (ب-7 وب-7-ب) بتقلل الشك، بس مابتلغيهوش. ممكن يكون فيه اختصارات تانية ما اختبرناهاش، زي علامات الجراحة، أو العلامة اللي بتتحط على الجلد مكان الكتلة قبل الأشعة.</li>
<li>اخترنا طريقة المربعات بعد ما قارناها بالصورة كلها، فرقمها ممكن يكون متفائل شوية.</li>
<li>ResNet50 متدربة على صور عادية (ImageNet)، مش صور أشعة.</li>
<li>تقارير الـ PDF مش مستخدمة لأنها مكتوب فيها الـ BI-RADS، يعني الإجابة نفسها (target leakage).</li>
</ul>
</div>

<div dir="rtl">

<h1>الجزء ج: ليه شلنا دمج التحاليل مع الصور</h1>
<ul>
<li>تحاليل Coimbra لـ 116 سيدة في البرتغال، وصور BCMID لـ 323 مريضة في الإسكندرية. مفيش ولا سيدة ليها تحاليل وصور مع بعض.</li>
<li>الدمج (Fusion) معناه نجمع معلومات عن <b>نفس المريضة</b>. لما نحط احتمال تحاليل سيدة جنب احتمال صور سيدة تانية، الناتج ملوش معنى طبي، وأي دقة بتطلع منه مش حقيقية.</li>
<li>السؤالين مختلفين كمان: Coimbra بتقارن مريضات بسيدات سليمات، وBCMID بتقارن درجات BI-RADS.</li>
<li><b>الدمج الصح</b> عندنا هو بين الماموجرام والسونار لنفس الحالة في BCMID، وده موجود في الجزء ب.</li>
<li><b>في التطبيق:</b> الدكتورة بتشوف كل مؤشر لوحده، ومعاه أداؤه المحسوب. التطبيق مش بيجمعهم في «تشخيص» واحد.</li>
</ul>

</div>

<div dir="rtl">
<h1>الملخص (محسوب من الخلايا اللي فوق)</h1>
</div>

In [ ]:
def fmt(m, s):
    return f'{m:.2f} ± {s:.2f}'


summary = [{'النموذج': 'تحاليل الدم: SVM بـ 4 تحاليل', 'البيانات': f'Coimbra، {len(df)} سيدة',
            'AUC': fmt(chosen['AUC'], chosen['AUC sd']),
            'Sensitivity': fmt(chosen['Sensitivity'], chosen['Sensitivity sd']),
            'Specificity': fmt(chosen['Specificity'], chosen['Specificity sd']),
            'Accuracy': fmt(chosen['Accuracy'], chosen['Accuracy sd'])}]
for name, label in [('Mammogram', 'ماموجرام'), ('Ultrasound', 'سونار'), ('Fusion', 'ماموجرام + سونار')]:
    summary.append({'النموذج': f'الأشعة: {label}', 'البيانات': f'BCMID، {len(cases)} حالة',
                    'AUC': fmt(img_mean.loc[name, 'AUC'], img_sd.loc[name, 'AUC']),
                    'Sensitivity': fmt(img_mean.loc[name, 'Sensitivity'], img_sd.loc[name, 'Sensitivity']),
                    'Specificity': fmt(img_mean.loc[name, 'Specificity'], img_sd.loc[name, 'Specificity']),
                    'Accuracy': fmt(img_mean.loc[name, 'Accuracy'], img_sd.loc[name, 'Accuracy'])})
display(pd.DataFrame(summary))
print(f'\n⚠️ للمقارنة: نموذج مابيشوفش الصور خالص (معلومات جانبية بس) جاب AUC = {fmt(*side_rf)} على نفس الحالات (ب-7-ب).')
print('الـ AUC جوه المجموعات (ب-7-ب):')
display(within)

<div dir="rtl">

<h1>الجزء د: أسئلة متوقعة من الدكتور</h1>

<p><b>س1: ليه ما اعتمدتيش على الـ Accuracy لوحدها؟</b><br>
في BCMID حوالي 80% من الحالات Negative. نموذج بيقول «Negative» على طول هياخد Accuracy حوالي 80% ومش هيكتشف ولا حالة. علشان كده بعرض AUC وSensitivity وSpecificity.</p>

<p><b>س2: يعني إيه Cross-Validation، وليه كررتيه؟</b><br>
بقسم البيانات 5 أجزاء، وبدرّب على 4 وأختبر على الخامس، وألف على الأجزاء كلها، وبعدين أكرر بتقسيمات مختلفة. مع 116 حالة بس، اختبار واحد على 24 حالة بيتأثر بالحظ جدًا.</p>

<p><b>س3: يعني إيه Data Leakage؟</b><br>
إن معلومة من بيانات الاختبار توصل للتدريب. مثال من نسختي القديمة: SelectKBest اختار الميزات وهو شايف تصنيف كل البيانات، ومنها بيانات الاختبار. الحل إن كل خطوة تبقى جوه Pipeline تتعلم من التدريب بس.</p>

<p><b>س4: ليه قسمتي BCMID حسب المريضة؟</b><br>
فيه 9 مريضات ليهم حالتين. لو حالة في التدريب والتانية في الاختبار، النموذج «يعرف» المريضة، ونتيجته تطلع أعلى من الحقيقة. StratifiedGroupKFold بيحط كل حالات المريضة في نفس الجزء.</p>

<p><b>س5: ليه ResNet50 المدربة على ImageNet؟</b><br>
البيانات صغيرة (حوالي 330 حالة) ومش كفاية تدرّب شبكة من الصفر. فبستخدم الشبكة كمستخرج ميزات (Transfer Learning)، وبعدها نموذج بسيط. ده خط أساس، والخطوة الجاية fine-tuning أو نماذج متخصصة في الثدي.</p>

<p><b>س6: ليه PCA؟</b><br>
كل حالة عندها 2048 رقم، وفي كل fold عندي حوالي 260 حالة تدريب بس. الـ PCA بيختصرهم لـ 50 علشان النموذج مايحفظش البيانات (overfitting)، وهو كمان جوه الـ Pipeline، فبيتعلم من التدريب بس.</p>

<p><b>س7: ليه قسمتي الماموجرام لمربعات؟</b><br>
لأن الصورة الأصلية حوالي 5000 بكسل، ولما بتتصغّر لـ 224 الورم الصغير والتكلسات بيختفوا. فكبّرتها لـ 672 وقسمتها 9 مربعات، وخدت أقوى إشارة في أي مربع وفي أي صورة. والنتيجة بالمربعات طلعت أحسن بوضوح من الصورة كلها (قارني جدول 1 وجدول 2 في ب-5). بس فيه تحفظين: أنا اخترت الطريقة دي بعد ما شفت النتيجتين، فرقمها ممكن يكون متفائل شوية. وكمان أخد أقوى إشارة من كل صور الحالة بيخلي النتيجة تتأثر بعدد الصور. ولما اختبرت ده في ب-7-ب، الماموجرام نزل كتير لما قارنت حالات بنفس عدد الصور، فجزء كبير من تفوق المربعات كان جاي من هنا.</p>

<p><b>س8: ليه ما دمجتيش التحاليل مع الصور؟</b><br>
لأنهم ستات مختلفين من بلدين مختلفين، ومفيش ولا مريضة ليها الاتنين. الدمج الصح عندي بين الماموجرام والسونار لنفس الحالة.</p>

<p><b>س9: تصنيف BCMID ده تشخيص نهائي؟</b><br>
لأ. ده تقييم BI-RADS من طبيب الأشعة، مش نتيجة عيّنة. فالنموذج بيتعلم يقلّد تقييم طبيب الأشعة. وBI-RADS 6 حالات سرطان متأكد قبل كده، وعلشان كده عملت اختبار من غيرهم.</p>

<p><b>س10: ينفع النماذج دي تتستخدم مع مريضات بجد؟</b><br>
لسه. دي نتايج داخلية على بيانات صغيرة من مكان واحد. محتاجين اختبار على بيانات من مستشفى تاني (external validation)، وموافقات أخلاقية، قبل أي استخدام.</p>

<p><b>س11: ليه مش كل الصور على الدرايف؟</b><br>
مساحة جوجل المجانية 15 GB، والداتا لوحدها حوالي 13 GB. فنزّلت الداتا من مصدرها الرسمي (Zenodo) على هارد Colab نفسه، وحسبت ميزات الصور الناقصة بنفس دوال التجهيز ونفس ResNet50 بالحرف، وحفظت الميزات بس، وحجمها كام ميجا بدل حوالي 4 جيجا صور. الصورة الخام بنحتاجها مرة واحدة بس علشان نطلع منها الميزات، وبعد كده النموذج بيشتغل على الميزات. وقايمة الصور كلها جاية من الفهرس الرسمي لملف الداتا، فمفيش ولا صورة اتسابت.</p>

<p><b>س12: إزاي اتأكدتي إن النموذج مش بيغش؟</b><br>
عملت 4 حاجات. الأولى: التقسيم حسب المريضة، فمفيش مريضة في التدريب والاختبار مع بعض. التانية: كل خطوة بتتعلم من البيانات (التطبيع والـ PCA) جوه Pipeline، فبتتعلم من التدريب بس. التالتة: شلت BI-RADS 6 وقيّمت تاني (ب-7)، لأن الحالات دي فيها سرطان متأكد قبل كده وعلامات علاج. الرابعة (ب-7-ب): اختبرت الاختصارات بشكل مباشر، ولقيت فعلًا اختصار في الداتا: نموذج مابيشوفش الصور خالص، وبيعرف بس عدد الصور والسنة، جاب AUC قد نموذج الصور تقريبًا. فقارنت حالات بنفس عدد الصور ونفس الفترة: النموذج اللي من غير صور نزل لقريب من 0.5، والسونار والدمج فضلوا كويسين، والماموجرام نزل كتير. يعني الصور فيها معلومة حقيقية، أغلبها من السونار، ورقم الماموجرام لوحده متفائل. والخطوة الجاية اختبار على داتا من مستشفى تانية (CDD-CESM)، وGrad-CAM علشان نشوف النموذج بيبص فين في الصورة.</p>

<p><b>س13: إيه الخطوة الجاية؟</b><br>
fine-tuning على الصور، وGrad-CAM علشان نشوف النموذج بيبص فين في الصورة، واستخدام بيانات فيها نتيجة العيّنة (زي CDD-CESM، اللي فيها عمود Pathology Classification)، واختبار على بيانات خارجية.</p>

</div>